In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
from runner import DualRunner

PG_CONNINFO = (
    f"host=127.0.0.1 "
    f"port={os.getenv("POSTGRES_PORT", 5432)} "
    f"dbname={os.getenv("POSTGRES_DB")} "
    f"user={os.getenv("POSTGRES_USER")} "
    f"password={os.getenv("POSTGRES_PASSWORD")}"
)

runner = DualRunner(
    pg_conninfo=PG_CONNINFO,
    duckdb_path=":memory:"
)

display(runner.run_pg("select version()"))
display(runner.run_dd("select version()"))

,version
0,PostgreSQL 17.7 (Debian 17.7-3.pgdg13+1) on aa...


,"""version""()"
0,v1.4.3


# データ加工のためのSQL
## 一つの値に対する処理

In [4]:
runner.check("""--sql
             
drop table if exists access_log;

create table access_log (
    stamp timestamp,
    referrer text,
    url text
);

insert into access_log (stamp, referrer, url) values
('2016-08-26 12:02:00', 'http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1', 'http://www.example.com/video/detail?id=001'),
('2016-08-26 12:02:01', 'http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1', 'http://www.example.com/video#ref'),
('2016-08-26 12:02:01', 'https://www.other.com/', 'http://www.example.com/book/detail?id=002');            

select * from access_log;
             
""")

same


,stamp,referrer,url
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=...,http://www.example.com/video/detail?id=001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=...,http://www.example.com/video#ref
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002


In [ ]:
runner.run_pg("""--sql

-- referrer から要素を取り出す            
select
    stamp,
    referrer,
    substring(referrer from 'https?://([^/]*)') as referrer_domain
from access_log
""")

runner.run_dd("""--sql
              
select
    stamp,
    referrer,
    substring(referrer from 'https?://([^/]*)') as referrer_domain
from access_log
              
""")

,stamp,referrer,referrer_domain
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=...,www.other.com
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=...,www.other.net
2,2016-08-26 12:02:01,https://www.other.com/,www.other.com
